# Notebook 1: Introduktion & Explorativ Dataanalys (EDA)

---

Detta är första steget i projektet. Innan vi bygger några modeller måste vi **förstå datan** –
vem är patienterna, hur ser variablerna ut, och finns det redan synliga mönster?

EDA är grunden för alla välgrundade beslut vi tar senare i projektet.

## Bakgrund

Hjärtsjukdom är en av de vanligaste dödsorsakerna globalt. Med maskininlärning kan vi
bygga modeller som hjälper kliniker att identifiera riskpatienter tidigt – baserat på
mätvärden som redan tas vid en vanlig läkarundersökning.

**Projektets mål:** Träna klassificeringsmodeller som förutsäger om en patient har hjärtsjukdom
utifrån 11 kliniska variabler. Vi jämför flera metoder och utvärderar hur väl de presterar.

## Dataset

**Källa:** fedesoriano  
**Plattform:** Kaggle  
**Länk:** https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction  
**Rader:** 918 patienter  
**Kolumner:** 12 (11 input-variabler + 1 målvariabel)  
**Målvariabel:** `HeartDisease` (0 = frisk, 1 = hjärtsjukdom)

## Variabelbeskrivning

| Variabel | Typ | Beskrivning |
|---|---|---|
| Age | Numerisk | Patientens ålder i år |
| Sex | Kategorisk | Kön: M (man) eller F (kvinna) |
| ChestPainType | Kategorisk | Bröstsmärtstyp: ATA, NAP, ASY, TA |
| RestingBP | Numerisk | Viloblodtryck i mm Hg |
| Cholesterol | Numerisk | Serumkolesterol i mm/dl |
| FastingBS | Binär | Fasteblodsocker > 120 mg/dl (1 = ja, 0 = nej) |
| RestingECG | Kategorisk | EKG vid vila: Normal, ST, LVH |
| MaxHR | Numerisk | Maximal uppmätt hjärtfrekvens |
| ExerciseAngina | Kategorisk | Angina vid träning: Y (ja) eller N (nej) |
| Oldpeak | Numerisk | ST-sänkning vid träning (relativ till vila) |
| ST_Slope | Kategorisk | Lutning på ST-segmentet: Up, Flat, Down |
| **HeartDisease** | **Binär** | **Målvariabel – 0 = frisk, 1 = sjuk** |

## Frågeställning

Kan vi förutsäga hjärtsjukdom utifrån kliniska mätvärden?

Specifika frågor:
1. Vilka variabler hänger starkast samman med hjärtsjukdom?
2. Finns det tydliga mönster som skiljer sjuka från friska patienter?
3. Hur väl kan en maskininlärningsmodell klassificera rätt?
4. Kan modellen förbättras med dimensionsreduktion (PCA/UMAP)?

---

## Ladda data

Vi börjar med att importera de bibliotek vi behöver och läsa in datasetet.
Ett bra första steg är alltid att titta på de första raderna och kontrollera formen.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Gemensamma utseendeinställningar för alla plottar i projektet
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

# Läs in rådata – vi pekar alltid på originalfilen, aldrig en kopia
df = pd.read_csv('Data/raw_heart_disease.csv')

print(f'Datasetet innehåller {df.shape[0]} patienter och {df.shape[1]} variabler.')
df.head()

## Datakvalitet

Innan vi analyserar måste vi kontrollera att datan är pålitlig:
- **Datatyper** – är varje kolumn lagrad som rätt typ (int, float, text)?
- **Saknade värden** – finns det tomma celler som kan störa analysen?
- **Dubbletter** – finns det identiska rader som räknas dubbelt?

In [ ]:
# Sammanfattning av datatyper och saknade värden per kolumn
kvalitet = pd.DataFrame({
    'Datatyp': df.dtypes,
    'Saknade värden': df.isnull().sum(),
    'Unika värden': df.nunique()
})
print(kvalitet.to_string())
print(f'\nAntal dubbletter: {df.duplicated().sum()}')

> **Fynd:** Inga saknade värden och inga dubbletter – bra datakvalitet!
>
> Vi noterar att `Cholesterol` och `RestingBP` kan innehålla värdet 0,
> vilket är medicinskt omöjligt. Det hanterar vi som ett outlier-problem i Notebook 2.

## Deskriptiv statistik

Vi sammanfattar fördelningen av varje variabel med enkla mätvärden:
medelvärde, standardavvikelse, min/max och kvartiler.

In [ ]:
print('=== Numeriska variabler ===')
df.describe().round(2)

In [ ]:
# Kategoriska variabler – hur många patienter finns i varje kategori?
kategoriska = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

for kol in kategoriska:
    print(f'--- {kol} ---')
    print(df[kol].value_counts().to_string())
    print()

## Klassbalans

Vi kontrollerar hur många patienter som är sjuka respektive friska.
Om en klass dominerar kraftigt kan modeller bli partiska mot den vanligare klassen.

In [ ]:
antal = df['HeartDisease'].value_counts().sort_index()
andel = df['HeartDisease'].value_counts(normalize=True).sort_index() * 100

print(f'Frisk (0): {antal[0]} patienter  ({andel[0]:.1f}%)')
print(f'Sjuk  (1): {antal[1]} patienter  ({andel[1]:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Frisk (0)', 'Sjuk (1)'], antal.values, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Klassbalans – HeartDisease', fontweight='bold')
ax.set_ylabel('Antal patienter')
for i, v in enumerate(antal.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/01_klassbalans.png', dpi=150)
plt.show()

> **Fynd:** Klassbalansen är relativt jämn (~45% friska, ~55% sjuka).
> Vi behöver inte vidta specialåtgärder (t.ex. oversampling) för obalans.

## Fördelningar

Histogram visar formen på varje variabels fördelning – är den normalfördelad,
skev, eller har den ovanliga toppar? Det hjälper oss att hitta outliers visuellt.

In [ ]:
numeriska = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, kol in enumerate(numeriska):
    ax = axes[i // 3][i % 3]
    ax.hist(df[kol], bins=30, color='steelblue', edgecolor='white')
    ax.set_title(kol)
    ax.set_xlabel('Värde')
    ax.set_ylabel('Frekvens')
axes[1][2].set_visible(False)  # Dölj den extra subploten
plt.suptitle('Fördelningar – numeriska variabler', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/01_fordelningar.png', dpi=150)
plt.show()

# Flagga de medicinsk omöjliga värdena vi såg i Datakvalitets-steget
print(f'Antal Cholesterol = 0: {(df["Cholesterol"] == 0).sum()}  ← troligen saknat värde')
print(f'Antal RestingBP  = 0: {(df["RestingBP"] == 0).sum()}  ← troligen saknat värde')

## Samband med målvariabeln

Här undersöker vi om det finns statistiskt signifikanta skillnader mellan
sjuka (1) och friska (0) patienter.

- **Numeriska variabler:** Boxplots + t-test (jämför medelvärden)
- **Kategoriska variabler:** Stapeldiagram med andel sjuka per kategori

In [ ]:
# Boxplots – visar spridning och median för sjuka vs friska
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, kol in enumerate(['Age', 'MaxHR', 'Oldpeak']):
    sns.boxplot(data=df, x='HeartDisease', y=kol, ax=axes[i],
                palette={0: 'steelblue', 1: 'tomato'})
    axes[i].set_title(f'{kol} vs HeartDisease')
    axes[i].set_xlabel('HeartDisease  (0 = Frisk, 1 = Sjuk)')
plt.suptitle('Boxplots – numeriska variabler vs HeartDisease', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/01_boxplots.png', dpi=150)
plt.show()

In [ ]:
# T-test: testar om medelvärdet skiljer sig signifikant mellan grupperna
# Nollhypotes: medelvärdet för friska = medelvärdet för sjuka
# Om p-värde < 0.05 → vi förkastar nollhypotesen → skillnaden är statistiskt signifikant

print(f'{'Variabel':<15}  {'p-värde':>10}   Slutsats')
print('-' * 52)
for kol in ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']:
    friska = df[df['HeartDisease'] == 0][kol]
    sjuka  = df[df['HeartDisease'] == 1][kol]
    _, p = stats.ttest_ind(friska, sjuka)
    markering = '✓ signifikant (p < 0.05)' if p < 0.05 else '✗ ej signifikant'
    print(f'{kol:<15}  {p:>10.4f}   {markering}')

In [ ]:
# Kategoriska variabler – andel sjuka per kategori
kategoriska_plot = ['Sex', 'ChestPainType', 'ExerciseAngina', 'ST_Slope']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, kol in enumerate(kategoriska_plot):
    ax = axes[i // 2][i % 2]
    # Beräkna andel sjuka (HeartDisease == 1) per kategori, i procent
    andel = df.groupby(kol)['HeartDisease'].mean() * 100
    andel.sort_values(ascending=False).plot(kind='bar', ax=ax,
        color='tomato', edgecolor='white', rot=0)
    ax.set_title(f'Andel sjuka per {kol} (%)', fontweight='bold')
    ax.set_ylabel('% patienter med hjärtsjukdom')
    ax.set_ylim(0, 105)
    for j, v in enumerate(andel.sort_values(ascending=False)):
        ax.text(j, v + 1.5, f'{v:.0f}%', ha='center', fontsize=11)
plt.suptitle('Kategoriska variabler vs HeartDisease', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/01_kategoriska_vs_target.png', dpi=150)
plt.show()

> **Fynd – samband med hjärtsjukdom:**
> - **MaxHR** och **Oldpeak** är starkt signifikanta (p << 0.05)
> - **Ålder** är signifikant – äldre patienter har högre risk
> - **ExerciseAngina = Y** (angina vid träning) är starkt kopplat till sjukdom
> - **ST_Slope = Flat** verkar vanligare bland sjuka patienter
> - **ChestPainType = ASY** (asymptomatisk) dominerar bland sjuka – farligt just för att det inte syns

## Korrelationsmatris

Korrelationsmatrisen visar linjära samband mellan numeriska variabler.
- Värde nära **+1** = stark positiv samvariation (båda ökar tillsammans)
- Värde nära **-1** = stark negativ samvariation (ena ökar, andra minskar)
- Värde nära **0** = inget linjärt samband

Hög korrelation *mellan* features kallas multikollinearitet och kan störa Logistisk Regression.

In [ ]:
numeriska_kols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'HeartDisease']
korr = df[numeriska_kols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(korr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Korrelationsmatris – numeriska variabler', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/01_korrelationsmatris.png', dpi=150)
plt.show()

> **Fynd:** De numeriska variablerna korrelerar inte kraftigt med varandra –
> multikollinearitet är inget stort problem för Logistisk Regression i detta dataset.
> `MaxHR` har en tydlig negativ korrelation med `HeartDisease` (−0.40):
> hög maxpuls = lägre risk.

## Sammanfattning EDA

### Vad lärde vi oss om patienterna?

1. **Datan är komplett** – inga saknade värden, men `Cholesterol = 0` och `RestingBP = 0`
   är datakvalitetsproblem som behandlas i nästa notebook.

2. **Balanserad klass** – ~45% friska och ~55% sjuka. Inga specialåtgärder behövs.

3. **Starka prediktorer** (troliga nyckelvariabler för modellerna):
   - `MaxHR` – sjuka patienter har lägre maxpuls
   - `Oldpeak` – sjuka har högre ST-sänkning
   - `ExerciseAngina` – angina vid träning är tydligt kopplat till sjukdom
   - `ST_Slope` – kurvan Flat/Down dominerar bland sjuka
   - `ChestPainType = ASY` – asymptomatisk smärta är det vanligaste mönstret hos sjuka

4. **Svagare prediktorer:** `RestingBP` och `Cholesterol` har svagare statistiskt samband.

---
**Nästa steg → Notebook 2: Dataförberedelse**
Vi rensar outliers, kodar kategoriska variabler och förbereder datan för modellering.